# M35 — Improve Retrieval

## WHOLE

M34 already grounds answers. The useful whole here is a **measured retriever**:

`frozen eval → baseline candidate ranking → worst queries → one change`

Change **one** of chunking, candidate k, or rerank. Keep the M34 labels
frozen. Cosine is not a relevance label. A better average is not a
better system if the critical slice still ranks a trap first.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a chunk id, a metric value, a corpus version,
or a named failure mode.

Do not download a reranker, do not edit `datasets/M34` after seeing
scores, and do not open Qdrant/HNSW. If a failure can be diagnosed from
candidate ids versus gold support ids, stay at that layer.

Canonical sources (named, not implemented): `sentence-transformers`,
`qdrant-docs`, `hnsw-paper`. Approximate indexes remain M36.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M35" / "retrieval_eval.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M33.semantic_search import (
    encode_query,
    load_canonical_corpus,
    load_canonical_index,
    search,
)
from missions.M35.retrieval_eval import (
    CANONICAL_CORPUS_VERSION,
    DEFAULT_CANDIDATE_K,
    EVAL_VERSION,
    RERANKER_IDENTITY,
    RERANKER_LEX,
    SYSTEM_MAP,
    ExperimentConfig,
    baseline_config,
    evaluate_set,
    generate_candidates,
    label_hash,
    leak_eval_phrasing,
    load_expected_payload,
    load_frozen_queries,
    load_query_map,
    load_transfer_payload,
    mean_reciprocal_rank,
    ndcg_at_k,
    project_labels,
    questions_sha256,
    recall_at_k,
    materialize,
    relabel_after_results,
    repair_eval_boundary,
    rerank_candidates,
    rescore_with_labels,
    round_metric,
    source_hash,
    worst_queries,
)

corpus = load_canonical_corpus()
index = load_canonical_index()
FROZEN = load_frozen_queries()
QUERY_MAP = load_query_map()
EXPECTED = load_expected_payload()

print("repository root:", ROOT)
print(SYSTEM_MAP)
print("eval_version:", EVAL_VERSION)
print("index_id:", index.metadata.index_id)
print("corpus_version:", corpus.version)
print("source_hash:", index.metadata.source_hash[:16])
print("model:", index.metadata.embedding.model, index.metadata.embedding.version)
print("downloaded:", index.metadata.downloaded, "network:", index.metadata.network_required)
print("questions:", len(FROZEN))


## MAP

```
frozen M34 labels --> FrozenQuery (eval_version, original chunk ids)
source documents --> versioned corpus (chunk size/overlap, source_hash)
versioned corpus --> M33 search / as_evidence --> candidate set (k)
candidate set --> identity | lex-overlap-v1 --> ranked ids (same members)
ranked ids x projected labels --> recall@k, MRR, nDCG@k
slices --> aggregate vs critical
```

**PREDICT** before every governed action. M36 still owns ANN/Qdrant/HNSW.
M32 still owns decoding. M34 still owns generation.


## Frozen evaluation set

`datasets/M34/questions.json` is the eval set (`m34.eval.v1`). This
mission may measure it. It may not rewrite support or relevance ids
after seeing a new ranker. Holdout ids are not a tuning set.


In [ ]:
print("eval_version", FROZEN[0].eval_version)
print("label_hash", label_hash(FROZEN))
print("questions_sha256", questions_sha256())
print("n", len(FROZEN), "holdout", [q.query_id for q in FROZEN if q.split == "holdout"])
for query_id in ("rag-reset-login", "rag-password-procedure", "rag-ticket-4412", "rag-h-invoice"):
    item = QUERY_MAP[query_id]
    print(query_id, "split", item.split, "support", item.support_chunk_ids, "relevant", item.relevant_chunk_ids)
    print(" ", item.text)


In [ ]:
ticket = QUERY_MAP["rag-ticket-4412"]
encoded = encode_query(ticket.text, query_id=ticket.query_id)
ticket_hits = search(
    index,
    encoded,
    top_k=3,
    live_corpus=corpus,
    query_id=ticket.query_id,
    enforce_freshness=True,
    enforce_provenance=True,
)
ticket_evidence = tuple(hit.as_evidence() for hit in ticket_hits.hits)
print("scored_candidates", ticket_hits.scored_candidates)
print("as_evidence keys", sorted(ticket_evidence[0]))
print("hit count", len(ticket_evidence), "k", 3)
for row in ticket_evidence:
    print(
        "rank",
        row["rank"],
        "index",
        row["index_id"],
        "hash",
        row["source_hash"][:12],
        "chars",
        len(row["text"]),
    )
print("MAP prints provenance, not ranked chunk ids")


## Predict before running — baseline candidate ranking

Timestamp a prediction before `run-baseline`.

Config: canonical M33 chunks, `candidate_k=3`, identity rerank, frozen
`m34.eval.v1` labels.

Predict:

- whether ticket `4412` ranks `doc-tickets::c0` or the neighbor `c1` first
- whether mean nDCG over answerable queries is 1.0
- whether `scored_candidates` equals corpus size or equals k

Do not treat the top cosine as a relevance label.


In [ ]:
baseline_cfg = baseline_config()
baseline_report = evaluate_set(config=baseline_cfg, queries=FROZEN, source_corpus=corpus)
print("config identity", baseline_cfg.identity())
print("eval_version", baseline_report.eval_version)
print("mean_recall", round_metric(baseline_report.mean_recall_at_k))
print("mean_mrr", round_metric(baseline_report.mean_mrr))
print("mean_ndcg", round_metric(baseline_report.mean_ndcg_at_k))
print("scored_candidates", baseline_report.scored_candidates, "proxy", baseline_report.mean_proxy_cost)
ticket_row = baseline_report.row_map()["rag-ticket-4412"]
print("ticket ranked_ids", ticket_row.ranked_ids)
print("ticket candidate_ids", ticket_row.candidate_ids)
print("ticket ndcg", round_metric(ticket_row.ndcg_at_k), "mrr", round_metric(ticket_row.mrr))
print("ticket failure", ticket_row.failure_mode)
password_row = baseline_report.row_map()["rag-password-procedure"]
print("password ranked_ids", password_row.ranked_ids)
print("password first_support_rank", password_row.first_support_rank)


In [ ]:
answerable = [row for row in baseline_report.rows if row.answerable]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ids = [row.query_id for row in answerable]
ndcgs = [row.ndcg_at_k for row in answerable]
mrrs = [row.mrr for row in answerable]
ax.bar([i - 0.18 for i in range(len(ids))], ndcgs, width=0.36, label="nDCG@3")
ax.bar([i + 0.18 for i in range(len(ids))], mrrs, width=0.36, label="MRR")
ax.set_xticks(range(len(ids)))
ax.set_xticklabels(ids, rotation=35, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("metric")
ax.set_title("Baseline cosine ranking on frozen M34 labels")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("per-query nDCG", list(zip(ids, [round_metric(v) for v in ndcgs])))


### Baseline is a ranking, not an answer

Ticket `4412` is answerable and the gold span is in the k=3 window, yet
the neighbor ticket can still sit at rank 1. Candidate recall can be
perfect while nDCG is not. That is the handoff from M34 retrieval misses
into a rank-sensitive metric.


## Predict before running — worst queries

Timestamp a prediction before `run-worst`.

Same baseline ranking. No new chunking, k, or rerank.

Predict the two lowest-nDCG answerable query ids and whether their
failure mode is `candidate_miss`, `trap_at_1`, or `ranking_miss`.


In [ ]:
worst = worst_queries(baseline_report.rows, n=4)
for row in worst:
    print(
        row.query_id,
        "ndcg", round_metric(row.ndcg_at_k),
        "mrr", round_metric(row.mrr),
        "mode", row.failure_mode,
        "ranked", row.ranked_ids,
        "support", row.support_ids,
    )


### Name the miss before changing a knob

If gold is outside the candidate set, raise k or change chunking. If
gold is inside and a trap is rank 1, rerank can help. Do not do both
at once.


## Predict before running — chunk size/overlap

Timestamp a prediction before `run-chunking`.

Named change: corpus version (`merged` vs `win32` vs `win48o16`).
Invariant: same source documents, frozen M34 labels (projected by span
overlap when ids change).

Predict:

- whether merged tickets become one mixed chunk
- whether mean nDCG rises because mixed chunks count as relevant
- whether `source_hash` stays the canonical M33 hash


In [ ]:
merged_cfg = ExperimentConfig(
    experiment_id="chunk-merged",
    corpus_version="m35.corpus.merged.v1",
    chunk_mode="merged",
    chunk_size=None,
    chunk_overlap=0,
    candidate_k=DEFAULT_CANDIDATE_K,
    reranker_id=RERANKER_IDENTITY,
)
win32_cfg = ExperimentConfig(
    experiment_id="chunk-win32",
    corpus_version="m35.corpus.win32.v1",
    chunk_mode="windows",
    chunk_size=32,
    chunk_overlap=0,
    candidate_k=DEFAULT_CANDIDATE_K,
    reranker_id=RERANKER_IDENTITY,
)
win48_cfg = ExperimentConfig(
    experiment_id="chunk-win48o16",
    corpus_version="m35.corpus.win48o16.v1",
    chunk_mode="windows",
    chunk_size=48,
    chunk_overlap=16,
    candidate_k=DEFAULT_CANDIDATE_K,
    reranker_id=RERANKER_IDENTITY,
)
merged_report = evaluate_set(config=merged_cfg, queries=FROZEN, source_corpus=corpus)
win32_report = evaluate_set(config=win32_cfg, queries=FROZEN, source_corpus=corpus)
win48_report = evaluate_set(config=win48_cfg, queries=FROZEN, source_corpus=corpus)
for name, report in (
    ("canonical", baseline_report),
    ("merged", merged_report),
    ("win32", win32_report),
    ("win48o16", win48_report),
):
    print(
        name,
        "version", report.corpus_version,
        "hash", report.source_hash[:12],
        "ndcg", round_metric(report.mean_ndcg_at_k),
        "identity", report.config_identity[:12],
    )
merged_ticket = merged_report.row_map()["rag-ticket-4412"]
print("merged ticket ranked", merged_ticket.ranked_ids)
print("merged ticket mixed", merged_ticket.mixed_ids)
print("label_hash unchanged", merged_report.label_hash == baseline_report.label_hash)
projected = project_labels(QUERY_MAP["rag-ticket-4412"], corpus, corpus)
print("canonical projection relevant", projected.relevant_ids)


In [ ]:
labels = ["canonical", "merged", "win32", "win48o16"]
vals = [
    baseline_report.mean_ndcg_at_k,
    merged_report.mean_ndcg_at_k,
    win32_report.mean_ndcg_at_k,
    win48_report.mean_ndcg_at_k,
]
fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.bar(labels, vals)
ax.set_ylim(0, 1.05)
ax.set_ylabel("mean nDCG@3 (answerable)")
ax.set_title("Same source docs, versioned chunking, frozen labels")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("chunk nDCG", list(zip(labels, [round_metric(v) for v in vals])))


### Larger chunks can hide a trap inside a relevant span

Merged tickets make `4412` easy to retrieve and mix `4413` into the
same object. That is not a free quality win. Labels were projected,
not rewritten.


## Predict before running — candidate k

Timestamp a prediction before `run-candidate-k`.

Named change: `candidate_k` 1 versus 5.
Invariant: canonical index/embeddings, identity rerank.

Predict:

- whether ticket gold is missing at k=1
- whether `scored_candidates` grows with k on this exact index
- whether candidate recall at k=5 is 1.0


In [ ]:
k1_cfg = ExperimentConfig(
    experiment_id="candidate-k1",
    corpus_version=CANONICAL_CORPUS_VERSION,
    chunk_mode="canonical",
    chunk_size=None,
    chunk_overlap=0,
    candidate_k=1,
    reranker_id=RERANKER_IDENTITY,
)
k5_cfg = ExperimentConfig(
    experiment_id="candidate-k5",
    corpus_version=CANONICAL_CORPUS_VERSION,
    chunk_mode="canonical",
    chunk_size=None,
    chunk_overlap=0,
    candidate_k=5,
    reranker_id=RERANKER_IDENTITY,
)
k1_report = evaluate_set(config=k1_cfg, queries=FROZEN, source_corpus=corpus)
k5_report = evaluate_set(config=k5_cfg, queries=FROZEN, source_corpus=corpus)
print("k1 scored", k1_report.scored_candidates, "candidate_recall", round_metric(k1_report.mean_candidate_recall))
print("k5 scored", k5_report.scored_candidates, "candidate_recall", round_metric(k5_report.mean_candidate_recall))
print("k1 ticket support_hit", k1_report.row_map()["rag-ticket-4412"].candidate_support_hit)
print("k5 ticket support_hit", k5_report.row_map()["rag-ticket-4412"].candidate_support_hit)
print("proxy k1", k1_report.mean_proxy_cost, "k5", k5_report.mean_proxy_cost)
print("k1 ids", k1_report.row_map()["rag-ticket-4412"].ranked_ids)


In [ ]:
ks = [1, 3, 5]
recalls = [
    k1_report.mean_candidate_recall,
    baseline_report.mean_candidate_recall,
    k5_report.mean_candidate_recall,
]
costs = [
    k1_report.mean_proxy_cost,
    baseline_report.mean_proxy_cost,
    k5_report.mean_proxy_cost,
]
fig, ax1 = plt.subplots(figsize=(6.5, 4.4))
ax1.plot(ks, recalls, marker="o", label="mean candidate recall")
ax1.set_xlabel("candidate k")
ax1.set_ylabel("candidate recall")
ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx()
ax2.plot(ks, costs, marker="s", color="C1", label="proxy_cost")
ax2.set_ylabel("latency proxy (scored + rerank k)")
ax1.set_title("Exact search scores the corpus; k truncates the list")
ax1.grid(alpha=0.3)
fig.tight_layout()
plt.show()
print("recall by k", list(zip(ks, [round_metric(v) for v in recalls])))
print("proxy by k", list(zip(ks, [round_metric(v) for v in costs])))


### Candidate recall is a ceiling for rerank

On this exact index, `scored_candidates` is the corpus size. k changes
the list handed downstream and the rerank cost, not how many vectors
were scored. ANN would change the scored set — that is M36.


## Predict before running — deterministic rerank

Timestamp a prediction before `run-rerank`.

Named change: `reranker_id=lex-overlap-v1`.
Invariant: the cosine candidate **set** is frozen (same members).

Predict:

- whether ticket ranked ids change while the id set stays equal
- whether invoice `99281` moves above the five-thousand neighbor
- whether mean MRR becomes 1.0


In [ ]:
rerank_cfg = ExperimentConfig(
    experiment_id="rerank-lex",
    corpus_version=CANONICAL_CORPUS_VERSION,
    chunk_mode="canonical",
    chunk_size=None,
    chunk_overlap=0,
    candidate_k=DEFAULT_CANDIDATE_K,
    reranker_id=RERANKER_LEX,
)
rerank_report = evaluate_set(config=rerank_cfg, queries=FROZEN, source_corpus=corpus)
ticket_cosine = generate_candidates(
    QUERY_MAP["rag-ticket-4412"].text,
    query_id="rag-ticket-4412",
    candidate_k=3,
    index=index,
    corpus=corpus,
)
ticket_rerank = rerank_candidates(
    ticket_cosine,
    reranker_id=RERANKER_LEX,
    query_text=QUERY_MAP["rag-ticket-4412"].text,
)
print("ticket cosine ids", ticket_cosine.ids())
print("ticket reranked ids", ticket_rerank.ids())
print("same members", ticket_cosine.id_set() == ticket_rerank.id_set())
print("invoice cosine", baseline_report.row_map()["rag-h-invoice"].ranked_ids)
print("invoice rerank", rerank_report.row_map()["rag-h-invoice"].ranked_ids)
print("password cosine", baseline_report.row_map()["rag-password-procedure"].ranked_ids)
print("password rerank", rerank_report.row_map()["rag-password-procedure"].ranked_ids)
print("mean ndcg cosine", round_metric(baseline_report.mean_ndcg_at_k), "lex", round_metric(rerank_report.mean_ndcg_at_k))
print("mean mrr cosine", round_metric(baseline_report.mean_mrr), "lex", round_metric(rerank_report.mean_mrr))


In [ ]:
crit = ["rag-ticket-4412", "rag-h-invoice", "rag-password-procedure", "rag-fifty", "rag-refund-deny"]
before = [baseline_report.row_map()[qid].ndcg_at_k for qid in crit]
after = [rerank_report.row_map()[qid].ndcg_at_k for qid in crit]
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(crit, before, marker="o", label="cosine / identity")
ax.plot(crit, after, marker="s", label="lex-overlap-v1")
ax.set_ylim(0, 1.05)
ax.set_ylabel("nDCG@3")
ax.set_title("Identical candidate sets; only order changes")
ax.set_xticks(range(len(crit)))
ax.set_xticklabels(crit, rotation=20, ha="right")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
print("critical nDCG before/after", list(zip(crit, [round_metric(v) for v in before], [round_metric(v) for v in after])))


### Rerank cannot invent a missing candidate

Ticket and invoice improve because gold was already in the window and
the lexical scorer uses digits / procedure cues. Password support
moves up without always becoming rank 1. That is still one named
change: the candidate generator did not run again.


## Predict before running — hard negatives

Timestamp a prediction before `run-hard-negatives`.

Named change: inject high-similarity irrelevant extras.
Invariant: frozen M34 relevant/support ids (the extras are not added
to gold).

Predict ticket `4412` top-1 after injection, and whether `label_hash`
stays equal to the freeze.


In [ ]:
hard_cfg = ExperimentConfig(
    experiment_id="hard-negatives",
    corpus_version=CANONICAL_CORPUS_VERSION,
    chunk_mode="canonical",
    chunk_size=None,
    chunk_overlap=0,
    candidate_k=DEFAULT_CANDIDATE_K,
    reranker_id=RERANKER_IDENTITY,
    hard_negatives=True,
)
hard_report = evaluate_set(config=hard_cfg, queries=FROZEN, source_corpus=corpus)
hard_ticket = hard_report.row_map()["rag-ticket-4412"]
print("hard ticket ranked", hard_ticket.ranked_ids)
print("hard ticket relevant", hard_ticket.relevant_ids)
print("hard trap_at_1", hard_ticket.trap_at_1, "ndcg", round_metric(hard_ticket.ndcg_at_k))
print("label_hash equal", hard_report.label_hash == baseline_report.label_hash)
print("scored_candidates", hard_report.scored_candidates)
hard_corpus, hard_index, _spec = materialize(hard_cfg, source_corpus=corpus, queries=FROZEN)
hard_candidates = generate_candidates(
    QUERY_MAP["rag-ticket-4412"].text,
    query_id="rag-ticket-4412",
    candidate_k=3,
    index=hard_index,
    corpus=hard_corpus,
)
print("hard candidate ids", hard_candidates.ids())
print("identity injection only; lex rerank is a later named change")


### A near-duplicate with the right digits is still a trap

The approval-wait extra is not in the frozen relevant list. Cosine
ranks it first. Labels stay frozen. Measuring lex on this same member
set is a later named change, not a second action in the injection.


## Predict before running — lex on the same hard-neg set

Timestamp a prediction before `run-hard-neg-rerank`.

Hold the ticket `4412` query, frozen labels, and the injected candidate
members. **Named change:** `reranker_id=lex-overlap-v1` on that same
set.

Predict:

- whether member sets stay equal to the identity injection
- whether lex moves gold `doc-tickets::c0` to rank 1 or the trap
  stays first


In [ ]:
hard_rerank = rerank_candidates(
    hard_candidates,
    reranker_id=RERANKER_LEX,
    query_text=QUERY_MAP["rag-ticket-4412"].text,
)
print("lex on hard-neg candidates", hard_rerank.ids())
print("same members", hard_candidates.id_set() == hard_rerank.id_set())
print("identity trap_at_1 still", hard_ticket.trap_at_1)


### Lex can tie and keep the trap

A digit-match scorer can tie gold with the approval extra and keep the
trap at rank 1 via the original cosine order. That is why hard
negatives belong in the eval, not in a silent relabel. The injection
cell did not apply this rerank.


## Predict before running — aggregate vs critical slice

Timestamp a prediction before `run-slices`.

Named change: the reporting slice (all answerable vs `critical`).
Invariant: the baseline ranking and labels.

Predict whether critical-slice nDCG is below the overall answerable
mean, and name one query that the average hides.


In [ ]:
print("aggregate ndcg", round_metric(baseline_report.slices["all"]["mean_ndcg_at_k"]))
print("critical ndcg", round_metric(baseline_report.slices["critical"]["mean_ndcg_at_k"]))
print("holdout ndcg", round_metric(baseline_report.slices["holdout_answerable"]["mean_ndcg_at_k"]))
print("unanswerable trap_at_1", round_metric(baseline_report.slices["unanswerable"]["trap_at_1_rate"]))
print("critical ids", baseline_report.slices["critical"]["query_ids"])
print("lex critical ndcg", round_metric(rerank_report.slices["critical"]["mean_ndcg_at_k"]))


In [ ]:
names = ["all answerable", "critical", "holdout answerable"]
base_vals = [
    baseline_report.slices["all"]["mean_ndcg_at_k"],
    baseline_report.slices["critical"]["mean_ndcg_at_k"],
    baseline_report.slices["holdout_answerable"]["mean_ndcg_at_k"],
]
lex_vals = [
    rerank_report.slices["all"]["mean_ndcg_at_k"],
    rerank_report.slices["critical"]["mean_ndcg_at_k"],
    rerank_report.slices["holdout_answerable"]["mean_ndcg_at_k"],
]
fig, ax = plt.subplots(figsize=(6.8, 4.4))
xpos = range(len(names))
ax.bar([x - 0.18 for x in xpos], base_vals, width=0.36, label="cosine")
ax.bar([x + 0.18 for x in xpos], lex_vals, width=0.36, label="lex rerank")
ax.set_xticks(list(xpos))
ax.set_xticklabels(names)
ax.set_ylim(0, 1.05)
ax.set_ylabel("mean nDCG@3")
ax.set_title("Averages hide the trap-heavy slice")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("slice table", list(zip(names, [round_metric(v) for v in base_vals], [round_metric(v) for v in lex_vals])))


### Ship the slice, not only the mean

Critical queries (ticket, invoice, password, fifty, refund) are where
neighbor traps live. Holdout stays unlabeled after these comparisons.


## Code reading — chunk, retrieve, label, rerank, score, identity

Read `rechunk_corpus`, `generate_candidates`, `project_labels`,
`rerank_candidates`, `ndcg_at_k`, `evaluate_set`, and
`ExperimentConfig.identity` in `missions/M35/retrieval_eval.py`.

**Predict before running** the next cell:

1. whether ticket cosine ids **equal** lex-reranked ids, or only the
   member set matches
2. whether `ndcg_at_k` returns a Python float computed here
3. whether a leaked corpus keeps a different `source_hash` from the
   repaired clean object

Probe objects (metric values, candidate ids vs reranked ids, version
ids). Do not search for HNSW or a temperature sampler.


In [ ]:
print(inspect.getsource(generate_candidates))
print(inspect.getsource(rerank_candidates))
print(inspect.getsource(ndcg_at_k))
print("candidate ids", ticket_cosine.ids())
print("reranked ids", ticket_rerank.ids())
print("member sets equal", ticket_cosine.id_set() == ticket_rerank.id_set())
print("orders equal", ticket_cosine.ids() == ticket_rerank.ids())
print("ticket ndcg object", baseline_report.row_map()["rag-ticket-4412"].ndcg_at_k)
print("ndcg is float", isinstance(baseline_report.row_map()["rag-ticket-4412"].ndcg_at_k, float))
print("config identity", baseline_cfg.identity())
print("corpus_version", baseline_report.corpus_version, "vs merged", merged_report.corpus_version)
print("recall_at_k toy", recall_at_k(["b", "a"], ["a"], 2))
print("mean_reciprocal_rank toy", mean_reciprocal_rank(["b", "a"], ["a"]))
print("ndcg_at_k toy", ndcg_at_k(["b", "a"], {"a": 2.0}, 2))
sample_evidence = ticket_cosine.items[0].evidence
print("as_evidence index_id", sample_evidence["index_id"])
print("as_evidence chunk_id", sample_evidence["chunk_id"])
leaked_preview = leak_eval_phrasing(corpus, FROZEN)
print("leaked source_hash", source_hash(leaked_preview)[:16])
print("clean source_hash", source_hash(corpus)[:16])
print("hashes equal", source_hash(leaked_preview) == source_hash(corpus))


## Predict before running — Controlled failure: eval leakage

Timestamp a prediction before `run-failure`.

Named change: `leak_eval_phrasing` copies eval query text into gold
support chunks. Labels stay frozen. Ranking code stays identity.

Predict whether mean nDCG rises, whether `source_hash` changes, and
whether the original `corpus` object still lacks the leaked query
string.


In [ ]:
leaked_corpus = leak_eval_phrasing(corpus, FROZEN)
leaked_cfg = ExperimentConfig(
    experiment_id="leak-eval-phrasing",
    corpus_version=CANONICAL_CORPUS_VERSION,
    chunk_mode="canonical",
    chunk_size=None,
    chunk_overlap=0,
    candidate_k=DEFAULT_CANDIDATE_K,
    reranker_id=RERANKER_IDENTITY,
    leaked=True,
)
leaked_report = evaluate_set(config=leaked_cfg, queries=FROZEN, source_corpus=corpus)
print("clean hash", source_hash(corpus)[:16])
print("leaked hash", source_hash(leaked_corpus)[:16])
print("clean still clean", "How do I reset my login credentials?" not in corpus.get_chunk("doc-account-access::c1").text)
print("leaked text", leaked_corpus.get_chunk("doc-account-access::c1").text)
print("ndcg clean", round_metric(baseline_report.mean_ndcg_at_k), "leaked", round_metric(leaked_report.mean_ndcg_at_k))
print("label_hash leaked run", leaked_report.label_hash == baseline_report.label_hash)


### Diagnose before repair

The average moved. Write a hypothesis: did ranking get better, or did
the eval set leak into the documents? Name a discriminating check
(source_hash, presence of the query string, frozen label_hash).


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Named change: `repair_eval_boundary` restores the clean corpus and
frozen labels. Do not relabel in this step.

Predict whether repaired nDCG matches baseline, and whether
`leaked_corpus` still contains the query string afterward.


In [ ]:
repaired_corpus, repaired_labels = repair_eval_boundary(
    broken_corpus=leaked_corpus,
    source_corpus=corpus,
    frozen_labels=FROZEN,
)
repaired_report = evaluate_set(config=baseline_cfg, queries=repaired_labels, source_corpus=repaired_corpus)
print("repaired hash", source_hash(repaired_corpus)[:16])
print("repaired equals clean", source_hash(repaired_corpus) == source_hash(corpus))
print("broken still leaked", "How do I reset my login credentials?" in leaked_corpus.get_chunk("doc-account-access::c1").text)
print("repaired ndcg", round_metric(repaired_report.mean_ndcg_at_k))
print("baseline ndcg", round_metric(baseline_report.mean_ndcg_at_k))
print("label_hash restored", label_hash(repaired_labels) == label_hash(FROZEN))


### Restore the boundary; leave the broken object broken

Repair does not mutate the leaked corpus. It returns the clean source
and the frozen labels. Metric jumps that required leaked phrasing are
not quality.


## Predict before running — metric gaming by relabeling

Timestamp a prediction before `run-relabel`.

Named change: after seeing ranks, mark top-1 as relevant on misses.
Invariant: ranked ids stay identical. This is **not** a second leak
repair.

Predict whether nDCG/MRR rise while ticket ranked ids stay
`c1, c0, ...`.


In [ ]:
gamed_labels = relabel_after_results(FROZEN, baseline_report.rows)
gamed_report = rescore_with_labels(
    baseline_report.rows,
    gamed_labels,
    config=baseline_cfg,
    corpus=corpus,
    index=index,
    source_corpus=corpus,
)
print("frozen hash", label_hash(FROZEN))
print("gamed hash", label_hash(gamed_labels))
print("ticket frozen relevant", QUERY_MAP["rag-ticket-4412"].relevant_chunk_ids)
print("ticket gamed relevant", next(q.relevant_chunk_ids for q in gamed_labels if q.query_id == "rag-ticket-4412"))
print("ranked ids unchanged", gamed_report.row_map()["rag-ticket-4412"].ranked_ids == baseline_report.row_map()["rag-ticket-4412"].ranked_ids)
print("ndcg frozen", round_metric(baseline_report.mean_ndcg_at_k), "gamed", round_metric(gamed_report.mean_ndcg_at_k))
print("relabeled flag", gamed_report.relabeled)


### Relabeling after results is not retrieval improvement

The ranking did not change. The metric changed because the labels
moved. M36 must inherit the frozen set, not this gamed copy.


## Evidence contract

Your evidence log must include timestamped predictions, the frozen
eval hash, baseline per-query metrics, worst-query failure modes, one
named change (chunking or k or rerank) with an invariant, candidate
ids versus reranked ids, aggregate versus critical slice, the leakage
repair, and the separate relabel demonstration.

Fixture numbers in `expected.json` are not learner evidence.


## No-AI gate

Close this notebook and complete `missions/M35/no_ai_gate.md` from a
blank page without AI-generated code, calculations, prose, or
diagrams.

Use only `datasets/M35/transfer.json`. Compute Recall@3 and nDCG@3,
diagnose one low-recall and one poor-ranking query, choose one lever,
spot the leakage example, and explain candidate recall versus final
ranking.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Author the V09 chunking/candidate/rerank policy in
`missions/M35/adr_prompt.md` using `templates/ADR.md`. Compare frozen
exact+lexical rerank with slice metrics, merge-and-relabel averages,
and a required model reranker plus Qdrant/HNSW.

**Status:** [UNFILLED BY LEARNER]

Formal engineering review uses `missions/M35/review_brief.md`. This
notebook is not a review signature.


## M34 → M35 handoff

M36 receives:

- frozen questions and support/relevance labels in `datasets/M34`
- a candidate/rerank interface (`generate_candidates`, `rerank_candidates`)
- measured exact-search quality and a latency proxy
- known slices (ticket/invoice traps, mixed merged chunks, hard negatives)

M36 may choose ANN/Qdrant/HNSW/hybrid infrastructure. It may not
silently relabel these questions. This notebook does not implement
those stacks; it only names `qdrant-docs` and `hnsw-paper` as the
sources to read next.


## Mission summary prompt

In your own words: what did freezing the eval set protect? Where did
candidate recall disagree with nDCG? Which single change would you
ship, and what slice would make you roll it back?

Leave the answers in your evidence log, not in this repository.


In [ ]:
assert FROZEN[0].eval_version == "m34.eval.v1"
assert questions_sha256() == EXPECTED["questions_sha256"]
assert label_hash(FROZEN) == EXPECTED["label_hash"]
assert baseline_report.row_map()["rag-ticket-4412"].ranked_ids[0] == "doc-tickets::c1"
assert rerank_report.row_map()["rag-ticket-4412"].ranked_ids[0] == "doc-tickets::c0"
assert ticket_cosine.id_set() == ticket_rerank.id_set()
assert leaked_report.mean_ndcg_at_k > baseline_report.mean_ndcg_at_k
assert round_metric(repaired_report.mean_ndcg_at_k) == EXPECTED["baseline"]["mean_ndcg_at_k"]
assert "How do I reset my login credentials?" in leaked_corpus.get_chunk("doc-account-access::c1").text
assert label_hash(gamed_labels) != label_hash(FROZEN)
assert baseline_report.slices["critical"]["mean_ndcg_at_k"] < baseline_report.slices["all"]["mean_ndcg_at_k"]
assert k1_report.mean_candidate_recall < k5_report.mean_candidate_recall
assert k1_report.scored_candidates == k5_report.scored_candidates
print("M35 integrity checks passed")
